## Running Accelerator Toolbox in a simplified view.

*Sverker Werin 260416*

IPAC 2026 version for Jupyter Notebook
(note! needs separate installation of acceleratortoolbox and ipywidgets a.o.)

In the Load and initialize section **Accelerator Toolbox** is installed the first time you run the script (once per session).

1. Start by running the complete Notebook (Windows: ctrl-F9 or from the menu: >Run all)
2. Open a module by pressing the ">" beside the title or press "cells hidden".
3. Calculations are made by push-buttons.
4. To edit or add a new lattice you need to open the "User lattices" cell in the "Latticemodule" (double-click the title. Double-click the title to close)
5. Running a cell is normally not necessary (Run a cell by pressing the arrow at the title or press shift-enter)


A template lattice is prepared by default "MAX I 550 MeV (template)". This is one cell of the MAX I storage ring.


# Load codes and initialize things (no need to open)

In [ ]:
#Load codes and initialize things (no need to open)

# 260427 This version is for ordinary Jupyter Notebook.
# All code is collected into the "Load codes and initialize things"
# section as function definitions.
# The interface is clean, but not as clean as when running the Colab
# version in Google Colab.
# 240830 Seems to be fixed, it works now
# 220927 Printing the complete matrix stopped working with AT 0.3.0.
# Quick fix: load lower version of AT.

#211025 Bug fixes with running AT 0.2.1
# Explicitly state PassMethod as default method changed.
# Optics calculation uses a lattice object instead of the lattice.

# Install Accelerator toolbox
!pip install accelerator-toolbox>=0.3.0
#!pip install 'accelerator-toolbox>=0.1.0,<0.3.0' # Force old version


# load libraries

import numpy as np
import at
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
from IPython.display import clear_output

# run this cell once for this Jupyter notebook.

# Setting some default values
length=1
xd=0
xpd=0
yd=0
ypd=0
dpd=0

# Define the lattice container
class latticeClass:
  name='myLattice'
  elements=pd.DataFrame(
    {
    1: ['Drift_empty', 0.0,0,0],
    2: ['Drift_empty', 0.0,0,0]
    },
    index=['type','length','bending angle','K-value']
)

# Define a template lattice
template=latticeClass()
template.name="MAX I 550 MeV (template)"
template.elements=pd.DataFrame(
    {
    1: ['Drift', 1.295, 0, 0],
    2: ['Quad',0.21,0,4.25],
    3: ['Drift',	0.265,	0,	0],
    4: ['Quad',	0.21,	0	,-3.02],
    5: ['Drift',	0.27025,	0,	0],
    6: ['Rbend',	0.9425,	0.785398,	0],
    7: ['Drift',	0.6523,	0	,0],
    8: ['Quad',	0.41,	0,	4.19],
    9: ['Drift',	0.6523,	0,	3],
    10: ['Rbend',	0.9425,	0.785398,	0],
    11: ['Drift',	0.27025,	0,	0],
    12: ['Quad',	0.21,	0,	-3.02],
    13: ['Drift',	0.265,	0,	0],
    14: ['Quad',	0.21,	0,	4.25],
    15: ['Drift',	1.295,	0,	0],
    },
    index=['type','length','bending angle','K-value']
)

#One cell of the MAX I accelerator
#The sample describes one section of the MAX I accelerator
lattice_template = pd.DataFrame(
    {
    1: ['Drift', 1.295, 0, 0],
    2: ['Quad',0.21,0,4.25],
    3: ['Drift',	0.265,	0,	0],
    4: ['Quad',	0.21,	0	,-3.02],
    5: ['Drift',	0.27025,	0,	0],
    6: ['Rbend',	0.9425,	0.785398,	0],
    7: ['Drift',	0.6523,	0	,0],
    8: ['Quad',	0.41,	0,	4.19],
    9: ['Drift',	0.6523,	0,	3],
    10: ['Rbend',	0.9425,	0.785398,	0],
    11: ['Drift',	0.27025,	0,	0],
    12: ['Quad',	0.21,	0,	-3.02],
    13: ['Drift',	0.265,	0,	0],
    14: ['Quad',	0.21,	0,	4.25],
    15: ['Drift',	1.295,	0,	0],
    },
    index=['type','length','bending angle','K-value']
)


def tunespace(tune,x_in,y_in,order): # Plot a tune space diagram
    plt.scatter(tune[0],tune[1],marker='o',s=100)

    x_off=x_in[0]
    x_min=x_in[0]-x_off
    x_max=x_in[1]-x_off

    y_off=y_in[0]
    y_min=y_in[0]-y_off
    y_max=y_in[1]-y_off

    F_0=np.maximum(x_max-x_min, y_max-y_min)


    if (order>4): # check for maximum order
        print('Maximum order 4')
        return

    # Define line type for resonance order
    line=['-b','--r','-.b',':g']


    for n in range(1,order+1): # Loop over orders
        ll=line[n-1] # find line type

        for i in range(0,n+1):
            a=i
            b=n-a
            for F in range(-F_0*n,F_0*n):

                if (a != 0):
                    xp=[(F-b*y_min)/a+x_off,(F-b*y_max)/a+x_off]
                    plt.plot(xp,y_in,ll)
                    xp=[(F+b*y_min)/a+x_off,(F+b*y_max)/a+x_off]
                    plt.plot(xp,y_in,ll)
                else:
                    yp=[F/b+y_off,F/b+y_off]
                    plt.plot(x_in,yp,ll)

    axes=plt.gca()
    axes.set_xlim(x_in)
    axes.set_ylim(y_in)
    axes.set_xlabel('Horisontal tune')
    axes.set_ylabel('Vertical tune')
    axes.set_title('Tune diagram (Blue dot = Working point)')

    plt.show()

    return

In [ ]:
def latticeSelector():
    #@title ###Select and make the lattice to use
    # Show radio buttons to select the lattice
    options=[]
    for i in range(len(Lattices)): # Find all lattices
      options.append(Lattices[i].name)

    latticeSelect=widgets.RadioButtons(
        options=options,
        description='Select lattice:',
        disabled=False
    )
    display(latticeSelect)

    # Select the periodicity of the lattice
    period=widgets.IntText(
        value=1,
        description='Periodicity =',
        disabled=False
    )

    display(period)

    #@title ###Make the lattice (update the cell!)

    # This module translates the input data in Pandas format to the format of
    # Accelerator Toolbox. If more types of elements are to be used, this has to
    # complemented.
    from IPython.display import clear_output

    selectedLattice=options.index(latticeSelect.value)
    button1L = widgets.Button(description="Make the Lattice")
    output1L = widgets.Output()

    display(button1L,output1L)
    lattice=0
    refpts=0
    def on_button1L_clicked(b):

        with output1L:
          clear_output()

          global lattice, lattice_t, num_of_elem # make the data global
          selectedLattice=options.index(latticeSelect.value)

          lattice_in=Lattices[selectedLattice].elements

          lattice_t=lattice_in.transpose()
          elements=lattice_t.values

          num_of_elem=len(lattice_in.columns)

          lattice=[at.elements.Drift('start', 0.00000)]
          # Loop over the entries in the lattice and extract relevant data
          for i in range(0,num_of_elem):
            #print(elements[i][0])
            if elements[i][0]=='Drift':
              elem=at.elements.Drift('drift',elements[i][1])
            elif elements[i][0]=='Quad':
              K=elements[i][3]
              elem=at.elements.Quadrupole('quad',elements[i][1],K)
              elem.PassMethod='QuadLinearPass'
            elif elements[i][0]=='Rbend':
              angle=elements[i][2]
              K=elements[i][3]
              elem=at.elements.Bend('dip',elements[i][1],angle)
              elem.K=K
              elem.EntranceAngle=angle/2
              elem.ExitAngle=angle/2
              elem.PassMethod='BendLinearPass'
            elif elements[i][0]=='Sbend':
              angle=elements[i][2]
              K=elements[i][3]
              elem=at.elements.Bend('dip',elements[i][1],angle)
              elem.K=K
              elem.EntranceAngle=0
              elem.ExitAngle=0
              elem.PassMethod='BendLinearPass'

            # Add element to the lattice list
            lattice=lattice+ [elem]

          # Repeat the lattice (periodicity)
          lattice=lattice[1:num_of_elem+1]*period.value

          num_of_elem=len(lattice) # update length of lattice
          print('Lattice ready with',num_of_elem,'elements.')

          # Define an AT lattice object
          global THERING, length, refpts,s, optics,beta,disp
          THERING = at.lattice.Lattice(lattice, energy=3.0, periodicity=period.value)
          length = np.size(THERING)
          refpts = np.r_[0:length + 1] # reference points for calculations and plot
          s = at.lattice.get_s_pos(THERING) # physical s position of elements

          # Calculate and check if the lattice has a closed solution
          try:
            optics = at.physics.linopt(THERING,refpts=refpts, get_chrom=True)
            beta = optics[3]['beta']
            disp = optics[3]['dispersion']
          except ValueError:
              print("Oops!  That was not a valid lattice.  Try again...")

    # Button to make the translation and basic optics check
    button1L.on_click(on_button1L_clicked)

In [ ]:
def displayLattice():
    #@title ###Display the lattice
    # Chose either comact of complete list of elements in the lattice
    from IPython.display import clear_output
    button1c = widgets.Button(description="Compact")
    output1c = widgets.Output()

    button1 = widgets.Button(description="Complete")
    output1 = widgets.Output()

    # Allow to clear the output as it can become long
    button1b = widgets.Button(description="Clear output")
    output1b = widgets.Output()

    display(widgets.HBox((button1c, button1, button1b)))
    display(output1)

    def on_button1_clicked(b):
        with output1:
          try:
            for i in range(0,num_of_elem):
              print(lattice[i])
          except NameError:
            print('No lattice defined')

    def on_button1c_clicked(b):
        with output1:
          try:
            print(lattice_t)
          except NameError:
            print('No lattice defined')

    def on_button1b_clicked(b):
        with output1:
          clear_output()


    button1.on_click(on_button1_clicked)
    button1c.on_click(on_button1c_clicked)
    button1b.on_click(on_button1b_clicked)


In [ ]:
def displayMatrices():
    #@title ### Matrices
    def on_button_clicked4(b): # print the 4x4 transfer matric of a defined element
        with output4:
          clear_output()
          element=w1.value
          try:
            m66=at.find_elem_m66(lattice[element-1])

            print('Transfer matrix of element',element,' = ')
            print(m66[0:4,0:4])
          except TypeError:
            print('Error')
          except IndexError:
            print('There are only',length,' elements')

    def on_button_clicked4b(b): # Print the transfer matrix of the complete lattice
        with output4b:
          clear_output()
          try:
            a=at.find_m44(THERING,0)
            print(a[0])
          except NameError:
            print('Error')


    button4 = widgets.Button(description="Print matrix of")
    output4 = widgets.Output()
    button4.on_click(on_button_clicked4)

    button4b = widgets.Button(description="Print the full lattice matrix")
    output4b = widgets.Output()
    button4b.on_click(on_button_clicked4b)

    w1=widgets.BoundedIntText(
        value=1,min=1, step=1,
        description='Element:',
        disabled=False
    )

    display(widgets.HBox((button4, w1)))
    display(output4)
    display(button4b, output4b)




In [ ]:
def displayGeometry():
    #@title ### Geometry
    # This module plots a simplified geometry of the elements alond a line.
    def on_button_clicked5(b):
        with output5:
          clear_output()
          try: # if a lattice exists
            # Define an array for coordinates of each element
            geometry=np.zeros((4,2*num_of_elem+2))


            for i in range (0,num_of_elem): # Loop the elements
              # make the s-axis with points at both entrance and exit
              # of each element
              geometry[0,2*i]=geometry[0,2*i-1]
              geometry[0,2*i+1]=lattice[i].Length+geometry[0,2*i]

              # Set values for different element types
              if lattice[i].FamName == 'quad':
                # Set flag for quad
                geometry[1,2*i]=1.5
                geometry[1,2*i+1]=1.5

                # Set the K value
                geometry[3,2*i]=lattice[i].K
                geometry[3,2*i+1]=lattice[i].K

              elif lattice[i].FamName == 'dip':
                geometry[1,2*i]=2
                geometry[1,2*i+1]=2

                # Set the K value
                geometry[3,2*i]=lattice[i].K
                geometry[3,2*i+1]=lattice[i].K

            # Scale the K values
            kMax=max(geometry[3,:])
            geometry[3,:]/=kMax

            #plt.plot(geometry[0], geometry[2],'r')
            plt.plot(geometry[0], geometry[3],'r')
            plt.plot(geometry[0], geometry[1],'b')
            plt.plot(geometry[0], -geometry[1],'b')


            axes = plt.gca()
            axes.set_xlabel('s (m)')
            axes.set_title('Geometry of lattice')
            plt.show()
          except NameError:
            print ('No lattice defined')


    button5 = widgets.Button(description="Plot geometry")
    output5 = widgets.Output()
    button5.on_click(on_button_clicked5)
    display(button5, output5)


In [ ]:
def calculateOptics():
    #@title ###Calculate the optics
    def on_button_clicked_optic(b):
        with output_optic:
          clear_output()

          try: # If a lattice and solution exists
            _, tunes, chroms, twiss = THERING.linopt(0.0, 0, get_chrom=True, coupled=False)
            tunes_full= optics[3]['mu'][length]/2/np.pi # Fix to get also integer tunes

            print('Tunes:', tunes_full)
            print('Circumference =', THERING.circumference)
            print('Chromaticity =', chroms)
          except NameError:
            print('No lattice defined')
          except ValueError:
            print('Not a valid (segement of a) ring')

    button_optic = widgets.Button(description="Calculate optics")
    output_optic = widgets.Output()

    button_optic.on_click(on_button_clicked_optic)

    display(button_optic,output_optic)



In [ ]:
def plotOptics():
    #@title ###Plot basic optic functions
    # Show selector for plots
    button = widgets.Button(description="Plot dispersion")
    output = widgets.Output()

    display(button, output)

    button2 = widgets.Button(description="Plot beta functions")
    output2 = widgets.Output()

    display(button2, output2)

    def on_button_clicked(b): # Plot the dispersion
        with output:
            clear_output()
            try: # check if lattice exists
              plt.plot(s, disp[:, 0], 'r')
              axes = plt.gca()
              axes.set_xlabel('s (m)')
              axes.set_ylabel('dispersion (m)')
              axes.set_title('Dispersion function')
              plt.show()
            except NameError:
              print('No lattice defined')

    def on_button_clicked2(b): # Plot the beta functions
        with output2:
          clear_output()
          try: # check if lattice exists
            l1, =plt.plot(s, beta[:, 0], color='green')
            l2, =plt.plot(s, beta[:, 1])
            axes = plt.gca()
            axes.set_xlabel('s (m)')
            axes.set_ylabel('beta (m)')
            axes.set_title('Beta functions')
            plt.legend([l1,l2],['Beta x','Beta y'])

            plt.show()
          except NameError:
            print('No lattice defined')

    button.on_click(on_button_clicked)
    button2.on_click(on_button_clicked2)

In [ ]:
def plotTuneSpace():
    #@title ###Plot tune space

    def on_button_clicked_ts(b):
        with output_ts:
          clear_output()
          try: # Check if lattice exists
            tunes_full=optics[3]['mu'][length]/2/np.pi
            tunespace(tunes_full,[x0.value,x1.value],[y0.value,y1.value],order.value)
          except NameError:
            print('No lattice defined')



    button_ts = widgets.Button(description="Plot tune space")
    output_ts = widgets.Output()

    button_ts.on_click(on_button_clicked_ts)

    # Show selector of plot limits
    x0=widgets.BoundedIntText(
        value=0,min=0,
        description='x min =',
        disabled=False
    )
    x1=widgets.BoundedIntText(
        value=2,min=0,
        description="x max =",
        disabled=False
    )
    y0=widgets.BoundedIntText(
    value=0,min=0,
    description='y min =',
    disabled=False
    )

    y1=widgets.BoundedIntText(
    value=2,min=0,
    description="y max =",
    disabled=False
    )

    #Show selector for resonsnce order
    order=widgets.BoundedIntText(
    value=2,min=0,max=4,
    description="Order =",
    disabled=False
    )
    display(button_ts)
    display(x0,x1,y0,y1,order)
    display(output_ts)

In [ ]:

def trackSingleParticle():
    #@title ###Track a single particle. Input particle coordinates.
    # Input of starting values for particle to track
    x=widgets.BoundedFloatText(
        value=0,min=0,
        description='x (mm) =',
        disabled=False
    )
    xp=widgets.BoundedFloatText(
        value=0,min=0,
        description="x' (mRad) =",
        disabled=False
    )
    y=widgets.BoundedFloatText(
    value=0,min=0,
    description='y (mm) =',
    disabled=False
    )

    yp=widgets.BoundedFloatText(
    value=0,min=0,
    description="y' (mRad) =",
    disabled=False
    )
    dp=widgets.BoundedFloatText(
    value=0,min=0,
    description="dp/p (0/00) =",
    disabled=False
    )

    display(x,xp,y,yp,dp)

    # Show action button to start tracking
    button7 = widgets.Button(description="Track and plot")
    output7 = widgets.Output()

    display(button7, output7)
    def on_button_clicked7(b):
        with output7:
          clear_output()
          try: #Check if lattice exists
            X0 = np.zeros((6,1))
            X0[0] = x.value*1e-3  # particle offset in x
            X0[1] = xp.value*1e-3  # particle offset in xp

            X0[2] = y.value*1e-3  # particle offset in y
            X0[3] = yp.value*1e-3  # particle offset in yp

            X0[4]=dp.value*1e-3

            # Track the particle in X0 through the lattice.
            X_out = at.lattice_pass(lattice, X0, nturns=1,refpts=refpts)
            plt.plot(s,X_out[0,0,:,0],'b')
            plt.plot(s,X_out[2,0,:,0],'r')
            axes = plt.gca()
            axes.set_xlabel('s (m)')
            axes.set_ylabel('offset (m)')
            axes.set_title('Particle position in x and y')
            plt.show()
          except TypeError:
            print('No valid lattice')
    button7.on_click(on_button_clicked7)




In [ ]:
def track1000Particles():

    #@title ###Track 1000 particles. Input particle distributions.
    # Input the rms values of the particles to track.
    # Either input values directly or calculate from given emittances
    # and the beta functions defined above. (Alpha is not taken into
    # account)

    print("Give input standard deviations")
    xd=widgets.BoundedFloatText(
        value=0,min=0,
        description='x rms (mm) =',
        disabled=False
    )
    xpd=widgets.BoundedFloatText(
        value=0,min=0,
        description="x' rms (mRad) =",
        disabled=False
    )
    yd=widgets.BoundedFloatText(
    value=0,min=0,
    description='y rms (mm) =',
    disabled=False
    )

    ypd=widgets.BoundedFloatText(
    value=0,min=0,
    description="y' rms (mRad) =",
    disabled=False
    )
    dpd=widgets.BoundedFloatText(
    value=0,min=0,
    description="dp/p rms (0/00) =",
    disabled=False
    )

    display(xd,xpd,yd,ypd,dpd)
    print('OR calculate from emittance and given optics')

    # Enter the emittances
    emit_x=widgets.BoundedFloatText(
    value=0,min=0,
    description="X emit (um) =",
    disabled=False
    )
    emit_y=widgets.BoundedFloatText(
    value=0,min=0,
    description="Y emit (um) =",
    disabled=False
    )

    display(emit_x,emit_y)
    # Show button to engage calculation of rms values from emittances
    print('Calculate input from emittances')
    button8 = widgets.Button(description="Calculate")
    output8 = widgets.Output()

    display(button8, output8)
    def on_button_clicked8(b):
        with output8:
          try:
            xd.value=np.sqrt(emit_x.value*1e-6*beta[0][0])
            xpd.value=np.sqrt(emit_x.value*1e-6/beta[0][0])

            yd.value=np.sqrt(emit_y.value*1e-6*beta[0][1])
            ypd.value=np.sqrt(emit_y.value*1e-6/beta[0][0])
          except NameError:
            print('No optics calculated')

    print('After defining input, track!')
    button8.on_click(on_button_clicked8)


    #@title ##### Generate the particles and track
    # Track the particles defined above
    global X_out_beam

    # Action button to engage tracking
    button9 = widgets.Button(description="Track and plot")
    output9 = widgets.Output()

    display(button9, output9)

    # Initialize variables

    i=1000 # number of particles

    # generate the particles from the rms values above
    def on_button_clicked9(b):
        global X_out_beam
        with output9:
          clear_output()


          #i=1000 # number of particles
          X2 = np.zeros((6, i))
          X2[0, :] = np.random.normal(0,1,i)*xd.value*1e-3  # particle offset in x
          X2[1, :] = np.random.normal(0,1,i)*xpd.value*1e-3  # particle offset in xp

          X2[2, :] = np.random.normal(0,1,i)*yd.value*1e-3  # particle offset in y
          X2[3, :] = np.random.normal(0,1,i)*ypd.value*1e-3  # particle offset in yp

          X2[4, :] = np.random.normal(0,1,i)*dpd.value*1e-3  # particle offset in yp

          try: # Try if tracking is correct
            # Track the particle in X0 through the lattice.
            X_out_beam = at.lattice_pass(lattice, X2, nturns=1,refpts=refpts)
            # Plot the real space. Show slider to step through the lattice
            interact(plot_rsp, element=widgets.IntSlider(min=0, max=num_of_elem, step=1,value=0));
            # Plot the phase space. Show slider to step through the lattice
            interact(plot_phsp, element=widgets.IntSlider(min=0, max=num_of_elem, step=1,value=0));
          except TypeError:
            print('Error in tracking')

    def plot_rsp(element):
      #Plot the real space
      plt.plot(X_out_beam[0,:,element,0],X_out_beam[2,:,element,0],'.')
      axes = plt.gca()
      axes.set_xlabel('x (m)')
      axes.set_ylabel('y (m)')
      axes.set_title('Real space')
      xmin=np.min(X_out_beam[0,:,:,0])
      xmax=np.max(X_out_beam[0,:,:,0])
      if xmin==xmax: # check to not get zero axis length
        xmin=xmin-0.05
        xmax=xmax+0.05
      xmax=np.max([np.abs(xmin),np.abs(xmax)])
      xmin=-xmax

      ymax=np.max(X_out_beam[2,:,:,0])
      ymin=np.min(X_out_beam[2,:,:,0])
      if ymin==ymax: # check to not get zero axis length
        ymin=ymin-0.05
        ymax=ymax+0.05
      ymax=np.max([np.abs(ymin),np.abs(ymax)])
      ymin=-ymax


      axes.set_xlim([xmin,xmax])
      axes.set_ylim([ymin,ymax])
      plt.show()

    def plot_phsp(element):
        #Plot phase space x - x'
        plt.figure(figsize=(8,4), dpi= 100, facecolor='w', edgecolor='k')
        plt.subplot(1,2,1)
        plt.plot(X_out_beam[0,:,element,0],X_out_beam[1,:,element,0],'.')
        axes = plt.gca()
        axes.set_xlabel('x (m)')
        axes.set_ylabel('xp (rad)')
        axes.set_title('Horisontal phase space')

        # Find the plot limits
        xmin=np.min(X_out_beam[0,:,:,0])
        xmax=np.max(X_out_beam[0,:,:,0])
        if xmin==xmax:
          xmin=xmin-0.05
          xmax=xmax+0.05
        xmax=np.max([np.abs(xmin),np.abs(xmax)])
        xmin=-xmax


        ymax=np.max(X_out_beam[1,:,:,0])
        ymin=np.min(X_out_beam[1,:,:,0])
        if ymin==ymax: # check to not get errors later
          ymin=ymin-0.05
          ymax=ymax+0.05
        ymax=np.max([np.abs(ymin),np.abs(ymax)])
        ymin=-ymax
        axes.set_xlim([xmin,xmax])
        axes.set_ylim([ymin,ymax])

        # Plot phase space y - y'
        plt.subplot(1,2,2)
        plt.plot(X_out_beam[2,:,element,0],X_out_beam[3,:,element,0],'.')
        axes = plt.gca()
        axes.set_xlabel('y (m)')

        # Find the plot limits
        xmin=np.min(X_out_beam[2,:,:,0])
        xmax=np.max(X_out_beam[2,:,:,0])
        if xmin==xmax:
          xmin=xmin-0.05
          xmax=xmax+0.05
        xmax=np.max([np.abs(xmin),np.abs(xmax)])
        xmin=-xmax

        ymax=np.max(X_out_beam[3,:,:,0])
        ymin=np.min(X_out_beam[3,:,:,0])
        if ymin==ymax: # check to not get errors later
          ymin=ymin-0.05
          ymax=ymax+0.05
        ymax=np.max([np.abs(ymin),np.abs(ymax)])
        ymin=-ymax
        axes.set_xlim([xmin,xmax])
        axes.set_ylim([ymin,ymax])

        axes.set_title('Vertical phase space')

        plt.show()

    button9.on_click(on_button_clicked9)

# Lattice module (open to edit lattices)

---



In [ ]:

#User lattices (Open to edit lattices)

Lattices=[template] # Do NOT change this line!
#
# Copy or change from HERE
# After adding/changing the lattice, re-run the notebook! (alt+F9 or "Run all")
#
myNewLattice=latticeClass()
myNewLattice.name='Empty lattice'
myNewLattice.elements = pd.DataFrame(
{
1: ['Drift', 2.2,0,0],
2: ['Quad',  0,0,0],
3: ['Sbend', 0,0,0] # no ',' on last element
},
index=['type','length','bending angle','K-value']
)
Lattices.append(myNewLattice) # Add the lattice to the list of lattices
#
# Copy and change UNTIL here
#

#
# Below is a lattice of the 3 GeV ring at MAX IV Laboratory (one achromat)
#
miv3gev=latticeClass()
miv3gev.name='MAX IV 3GeV'
miv3gev.elements = pd.DataFrame(
  {
  1: ['Drift', 2.5,0,0],
  2: ['Drift', 0.175,0,0],
  3: ['Quad', 0.25 ,0,3.533676],
  4: ['Drift', 0.225,0,0],
  5: ['Quad', 0.25 ,0,-2.23958],
  6: ['Drift', 0.006,0,0],
  7: ['Sbend', 0.75432, 1.5*(np.pi/180), -0.54525],
  8: ['Drift', 0.46268,0,0],
  9: ['Drift', 1.302,0,0],
  10: ['Quad', 0.55,0, 2.061576],
  11: ['Drift', 0.51311,0,0],
  12: ['Sbend', 1.22387, 3.0*(np.pi/180), -0.70480],
  13: ['Drift', 0.51311,0,0],
  14: ['Quad', 0.55 ,0,2.202441],
  15: ['Drift', 0.51311,0,0],
  16: ['Sbend', 1.22378, 3.0*(np.pi/180), -0.70408],
  17: ['Drift', 0.51311,0,0],
  18: ['Quad', 0.55 ,0,2.202441],
  19: ['Drift', 0.51311,0,0],
  20: ['Sbend', 1.22378, 3.0*(np.pi/180), -0.70408],
  21: ['Drift', 0.51311,0,0],
  22: ['Quad', 0.55 ,0,2.202441],
  23: ['Drift', 0.51311,0,0],
  24: ['Sbend', 1.22378, 3.0*(np.pi/180), -0.70408],
  25: ['Drift', 0.51311,0,0],
  26: ['Quad', 0.55,0, 2.202441],
  27: ['Drift', 0.51311,0,0],
  28: ['Sbend', 1.22378, 3.0*(np.pi/180), -0.70408],
  29: ['Drift', 0.51311,0,0],
  30: ['Quad', 0.55 ,0,2.061576],
  31: ['Drift', 1.302,0,0],
  32: ['Drift', 0.46268,0,0],
  33: ['Sbend', 0.75432 ,1.5*(np.pi/180) ,-0.54525],
  34: ['Drift', 0.006,0,0],
  35: ['Quad', 0.25 ,0,-2.23958],
  36: ['Drift', 0.225,0,0],
  37: ['Quad', 0.25 ,0, 3.533676],
  38: ['Drift', 0.175,0,0],
  39: ['Drift', 2.5,0,0]
  },
  index=['type','length','bending angle','K-value']
)
Lattices.append(miv3gev) # Add the lattice to the list of lattices


# Lattice Selection

---



In [ ]:
latticeSelector()

In [ ]:
displayLattice()

# Matrices and geometry module

---





In [ ]:
displayMatrices()

In [ ]:
displayGeometry()


# Optics module

---
    

In [ ]:
calculateOptics()

In [ ]:
plotOptics()

In [ ]:
plotTuneSpace()

# Tracking module

---



In [ ]:
trackSingleParticle()

In [ ]:
track1000Particles()

# End

---
